In [1]:
%%writefile /kaggle/working/extract_qwen_ppl.py
import argparse
import gc
import os
import time
from datetime import timedelta
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA_DIR = Path("/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026")
BASE_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/01-eda-split-baseline")
OUT_DIR = Path("/kaggle/working")

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
CLASSES = ["human", "fusion", "ai"]

FEAT_NAMES = [
    "qwen_n_tokens", "qwen_truncated",
    "qwen_nll_mean", "qwen_nll_std", "qwen_nll_q10", "qwen_nll_q25",
    "qwen_nll_q50", "qwen_nll_q75", "qwen_nll_q90", "qwen_nll_iqr",
    "qwen_ent_mean", "qwen_ent_std",
    "qwen_surp_mean", "qwen_surp_std", "qwen_nll_over_ent",
    "qwen_logrank_mean", "qwen_logrank_std", "qwen_top10_rate", "qwen_top100_rate",
    "qwen_win_nll_min", "qwen_win_nll_max", "qwen_win_nll_range",
    "qwen_win_nll_std", "qwen_win_nll_q10", "qwen_win_nll_q90",
    "qwen_win_nll_ratio", "qwen_win_nll_max_jump",
    "qwen_win_rank_min", "qwen_win_rank_max", "qwen_win_rank_range", "qwen_win_rank_std",
    "qwen_win_surp_min", "qwen_win_surp_max", "qwen_win_surp_range",
    "qwen_q1_nll", "qwen_q2_nll", "qwen_q3_nll", "qwen_q4_nll",
    "qwen_head_minus_tail_nll", "qwen_mid_minus_edges_nll",
]


def sliding_means(arr: np.ndarray, win: int = 64, stride: int = 32) -> np.ndarray:
    L = len(arr)
    if L <= win:
        return np.array([float(arr.mean())], dtype=np.float32)
    cs = np.empty(L + 1, dtype=np.float64)
    cs[0] = 0.0
    np.cumsum(arr, out=cs[1:])
    starts = np.arange(0, L - win + 1, stride)
    means = (cs[starts + win] - cs[starts]) / win
    if starts[-1] + win < L:
        tail_mean = (cs[L] - cs[L - win]) / win
        means = np.append(means, tail_mean)
    return means.astype(np.float32)


def compute_doc_features(seg_results, orig_n_tokens: int, max_len: int) -> np.ndarray:
    # seg_results: list of (nll_f32, ent_f32, rank_i32) in chronological order
    nll_all = np.concatenate([s[0] for s in seg_results]).astype(np.float32)
    ent_all = np.concatenate([s[1] for s in seg_results]).astype(np.float32)
    rank_all = np.concatenate([s[2] for s in seg_results]).astype(np.float32)
    surp_all = nll_all - ent_all
    logrank_all = np.log1p(rank_all)

    nll_mean = float(nll_all.mean())
    nll_std = float(nll_all.std())
    q10, q25, q50, q75, q90 = np.percentile(nll_all, [10, 25, 50, 75, 90])

    ent_mean = float(ent_all.mean())
    ent_std = float(ent_all.std())
    surp_mean = float(surp_all.mean())
    surp_std = float(surp_all.std())
    nll_over_ent = nll_mean / (ent_mean + 1e-6)

    logrank_mean = float(logrank_all.mean())
    logrank_std = float(logrank_all.std())
    top10_rate = float((rank_all < 10).mean())
    top100_rate = float((rank_all < 100).mean())

    # Вікна рахуємо окремо в кожному неперервному сегменті (без штучного шва)
    w_nll_list, w_rank_list, w_surp_list = [], [], []
    max_jump = 0.0
    for s_nll, s_ent, s_rank in seg_results:
        wn = sliding_means(s_nll, 64, 32)
        wr = sliding_means(np.log1p(s_rank.astype(np.float32)), 64, 32)
        ws = sliding_means((s_nll - s_ent), 64, 32)
        w_nll_list.append(wn)
        w_rank_list.append(wr)
        w_surp_list.append(ws)
        if len(wn) >= 3:
            max_jump = max(max_jump, float(np.abs(wn[2:] - wn[:-2]).max()))
        elif len(wn) == 2:
            max_jump = max(max_jump, float(abs(wn[1] - wn[0])))

    w_nll = np.concatenate(w_nll_list)
    w_rank = np.concatenate(w_rank_list)
    w_surp = np.concatenate(w_surp_list)

    wn_min, wn_max = float(w_nll.min()), float(w_nll.max())
    wn_q10, wn_q90 = np.percentile(w_nll, [10, 90])
    wr_min, wr_max = float(w_rank.min()), float(w_rank.max())
    ws_min, ws_max = float(w_surp.min()), float(w_surp.max())

    # Четверті документа
    quarters = np.array_split(nll_all, 4)
    q_means = [float(q.mean()) if len(q) > 0 else nll_mean for q in quarters]
    q1, q2, q3, q4 = q_means

    return np.array([
        float(orig_n_tokens),
        1.0 if orig_n_tokens > max_len else 0.0,
        nll_mean, nll_std, float(q10), float(q25), float(q50), float(q75), float(q90), float(q75 - q25),
        ent_mean, ent_std,
        surp_mean, surp_std, nll_over_ent,
        logrank_mean, logrank_std, top10_rate, top100_rate,
        wn_min, wn_max, wn_max - wn_min, float(w_nll.std()), float(wn_q10), float(wn_q90),
        wn_min / (wn_max + 1e-6), max_jump,
        wr_min, wr_max, wr_max - wr_min, float(w_rank.std()),
        ws_min, ws_max, ws_max - ws_min,
        q1, q2, q3, q4,
        q1 - q4, 0.5 * (q2 + q3) - 0.5 * (q1 + q4),
    ], dtype=np.float32)


@torch.no_grad()
def extract_split(
    texts,
    model,
    tokenizer,
    device,
    rank: int,
    world_size: int,
    max_len: int = 2048,
    max_tokens_per_batch: int = 4096,
    max_bs: int = 32,
    lm_chunk: int = 512,
    split_name: str = "val",
):
    n_total = len(texts)
    pad_id = tokenizer.pad_token_id
    eos_id = tokenizer.eos_token_id

    # Чергуємо індекси після сортування за довжиною -> рівне навантаження на обидва GPU
    char_order = np.argsort([len(t) for t in texts])
    local_doc_ids = char_order[rank::world_size]

    # Токенізація локальних документів і розбиття >max_len на head + tail
    segments = []
    orig_lengths = {}
    half = max_len // 2
    for doc_id in local_doc_ids:
        ids = tokenizer.encode(texts[int(doc_id)], add_special_tokens=False)
        orig_lengths[int(doc_id)] = len(ids)
        if len(ids) < 2:
            ids = [eos_id, eos_id]
        if len(ids) <= max_len:
            segments.append((int(doc_id), 0, ids))
        else:
            segments.append((int(doc_id), 0, ids[:half]))
            segments.append((int(doc_id), 1, ids[-half:]))

    # Сортуємо сегменти за довжиною для мінімального падінгу
    segments.sort(key=lambda x: len(x[2]))

    # Формуємо динамічні батчі з фіксованим бюджетом токенів
    batches = []
    cur = []
    for seg in segments:
        seg_len = len(seg[2])
        if cur and (len(cur) >= max_bs or (len(cur) + 1) * seg_len > max_tokens_per_batch):
            batches.append(cur)
            cur = [seg]
        else:
            cur.append(seg)
    if cur:
        batches.append(cur)

    doc_seg_store = {}
    backbone = model.model
    lm_head = model.lm_head
    t0 = time.time()
    tokens_done = 0

    for b_idx, batch in enumerate(batches, start=1):
        bs = len(batch)
        max_l = max(len(x[2]) for x in batch)
        input_ids = torch.full((bs, max_l), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((bs, max_l), dtype=torch.long)
        for i, (_, _, seq) in enumerate(batch):
            L = len(seq)
            input_ids[i, :L] = torch.tensor(seq, dtype=torch.long)
            attention_mask[i, :L] = 1

        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            hidden = backbone(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state

        h_shift = hidden[:, :-1, :]
        y_shift = input_ids[:, 1:]
        m_shift = attention_mask[:, 1:].bool()
        valid_counts = m_shift.sum(dim=1).cpu().tolist()

        h_flat = h_shift[m_shift]
        y_flat = y_shift[m_shift]
        n_valid = h_flat.size(0)
        tokens_done += n_valid

        nll_parts, ent_parts, rank_parts = [], [], []
        for s in range(0, n_valid, lm_chunk):
            h_c = h_flat[s : s + lm_chunk]
            y_c = y_flat[s : s + lm_chunk]
            logits = lm_head(h_c).float()
            log_probs = F.log_softmax(logits, dim=-1)
            nll = -log_probs.gather(1, y_c.unsqueeze(1)).squeeze(1)
            ent = -(log_probs.exp() * log_probs).sum(dim=-1)
            true_logits = logits.gather(1, y_c.unsqueeze(1))
            tok_rank = (logits > true_logits).sum(dim=-1)

            nll_parts.append(nll.half().cpu().numpy())
            ent_parts.append(ent.half().cpu().numpy())
            rank_parts.append(tok_rank.to(torch.int32).cpu().numpy())

        nll_cat = np.concatenate(nll_parts)
        ent_cat = np.concatenate(ent_parts)
        rank_cat = np.concatenate(rank_parts)

        offset = 0
        for (doc_id, seg_ord, _), cnt in zip(batch, valid_counts):
            sl = slice(offset, offset + cnt)
            doc_seg_store.setdefault(doc_id, []).append(
                (seg_ord, nll_cat[sl], ent_cat[sl], rank_cat[sl])
            )
            offset += cnt

        if rank == 0 and (b_idx % max(1, len(batches) // 10) == 0 or b_idx == len(batches)):
            dt = time.time() - t0
            vram = torch.cuda.max_memory_allocated(device) / 1024**3
            print(
                f"[{split_name}] batch {b_idx}/{len(batches)} | {tokens_done / max(dt, 1e-3):.0f} tok/s/gpu | "
                f"vram_peak={vram:.2f}GB | elapsed={dt:.1f}s",
                flush=True,
            )

    # Агрегуємо компактні фічі на документ
    local_feats = np.empty((len(local_doc_ids), len(FEAT_NAMES)), dtype=np.float32)
    for i, doc_id in enumerate(local_doc_ids):
        segs = sorted(doc_seg_store[int(doc_id)], key=lambda x: x[0])
        seg_arrays = [(s[1].astype(np.float32), s[2].astype(np.float32), s[3]) for s in segs]
        local_feats[i] = compute_doc_features(seg_arrays, orig_lengths[int(doc_id)], max_len)

    del doc_seg_store, segments
    gc.collect()

    gathered = [None] * world_size
    dist.all_gather_object(gathered, (local_doc_ids.astype(np.int64), local_feats))
    if rank != 0:
        return None

    full_feats = np.zeros((n_total, len(FEAT_NAMES)), dtype=np.float32)
    seen = np.zeros(n_total, dtype=bool)
    for g_ids, g_f in gathered:
        full_feats[g_ids] = g_f
        seen[g_ids] = True
    assert seen.all(), f"Missing {(~seen).sum()} docs!"
    assert np.isfinite(full_feats).all(), "Non-finite perplexity features detected!"
    return full_feats


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--smoke_test", action="store_true")
    parser.add_argument("--max_len", type=int, default=2048)
    parser.add_argument("--max_tokens_per_batch", type=int, default=4096)
    parser.add_argument("--lm_chunk", type=int, default=512)
    args = parser.parse_args()

    dist.init_process_group(backend="nccl", timeout=timedelta(minutes=60))
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)

    # Читаємо лише потрібні рядки (fold 0 + test)
    splits_df = pd.read_parquet(BASE_DIR / "folds.parquet")
    train_df = pd.read_csv(DATA_DIR / "train.csv", usecols=["id", "text", "label"])
    assert (train_df["id"].to_numpy() == splits_df["id"].to_numpy()).all()
    va_df = train_df[splits_df["fold"].to_numpy() == 0].reset_index(drop=True)
    del train_df, splits_df
    gc.collect()

    test_df = pd.read_csv(DATA_DIR / "test.csv", usecols=["id", "text"])
    full_va_len, full_te_len = len(va_df), len(test_df)

    if args.smoke_test:
        va_df = va_df.sample(n=1200, random_state=42).reset_index(drop=True)
        test_df = test_df.head(600).reset_index(drop=True)

    if rank == 0:
        mode = "SMOKE TEST" if args.smoke_test else "FULL RUN"
        print(f"[{mode}] {MODEL_NAME} | GPUs={world_size} | max_len={args.max_len} | Val={len(va_df)} | Test={len(test_df)}", flush=True)

    if rank != 0:
        dist.barrier()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
    if rank == 0:
        dist.barrier()

    model.eval()
    model.to(device)

    t_va = time.time()
    va_feats = extract_split(
        va_df["text"].tolist(), model, tokenizer, device, rank, world_size,
        max_len=args.max_len, max_tokens_per_batch=args.max_tokens_per_batch,
        lm_chunk=args.lm_chunk, split_name="val_fold0"
    )
    dt_va = time.time() - t_va

    t_te = time.time()
    te_feats = extract_split(
        test_df["text"].tolist(), model, tokenizer, device, rank, world_size,
        max_len=args.max_len, max_tokens_per_batch=args.max_tokens_per_batch,
        lm_chunk=args.lm_chunk, split_name="test"
    )
    dt_te = time.time() - t_te

    if rank == 0:
        vram_peak = torch.cuda.max_memory_allocated(device) / 1024**3
        va_out = pd.DataFrame(va_feats, columns=FEAT_NAMES)
        va_out.insert(0, "id", va_df["id"].to_numpy())
        te_out = pd.DataFrame(te_feats, columns=FEAT_NAMES)
        te_out.insert(0, "id", test_df["id"].to_numpy())

        # Sanity check розподілу ключових фіч по класах
        check = va_out[["qwen_nll_mean", "qwen_surp_mean", "qwen_win_nll_range", "qwen_win_nll_max_jump", "qwen_top10_rate"]].copy()
        check["label"] = va_df["label"].to_numpy()
        print("\n--- Sanity Check: Mean Qwen PPL features by class (Val Fold 0) ---")
        print(check.groupby("label").mean().reindex(CLASSES).round(4).to_string())
        print(f"\nPeak VRAM: {vram_peak:.2f}GB | Val time: {dt_va:.1f}s | Test time: {dt_te:.1f}s")

        if args.smoke_test:
            est_va = dt_va * (full_va_len / len(va_df)) / 60.0
            est_te = dt_te * (full_te_len / len(test_df)) / 60.0
            print(f"[ETA PROJECTION] Full Val ≈ {est_va:.1f} min | Full Test ≈ {est_te:.1f} min | Total ≈ {(est_va + est_te) / 60:.2f} h")
        else:
            va_out.to_parquet(OUT_DIR / "qwen_ppl_oof_fold0.parquet", index=False)
            te_out.to_parquet(OUT_DIR / "qwen_ppl_test.parquet", index=False)
            print(f"Saved: qwen_ppl_oof_fold0.parquet ({va_out.shape}), qwen_ppl_test.parquet ({te_out.shape})")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/extract_qwen_ppl.py


In [2]:
!torchrun --nproc_per_node=2 --master_port=29503 /kaggle/working/extract_qwen_ppl.py

W1123 11:52:16.755000 44 torch/distributed/run.py:851] 
W1123 11:52:16.755000 44 torch/distributed/run.py:851] *****************************************
W1123 11:52:16.755000 44 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1123 11:52:16.755000 44 torch/distributed/run.py:851] *****************************************
[FULL RUN] Qwen/Qwen2.5-0.5B | GPUs=2 | max_len=2048 | Val=54590 | Test=25000
config.json: 100%|█████████████████████████████| 681/681 [00:00<00:00, 2.43MB/s]
tokenizer_config.json: 100%|███████████████| 7.23k/7.23k [00:00<00:00, 10.5MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 65.1MB/s]
merges.txt: 100%|██████████████████████████| 1.67M/1.67M [00:00<00:00, 60.5MB/s]
tokenizer.json: 100%|██████████████████████| 7.03M/7.03M [00:00<00:00, 86.3MB/s]
[t